# 08 — pLLM supervised benchmark: ESM-embedding classifier (TEM-1 / Firnberg)

**Task.** Train a classifier on **ESM embeddings** (not scores) to predict functional vs non-functional
(`DMS_score_bin`), under the identical three-split battery as `02`/`06`, then run the **4-way comparison**:
AA-identity (`02`) vs physicochemical (`04`) vs zero-shot pLLM (`06`) vs **supervised-PLM (this notebook)**.

**Why this notebook exists (arm-3).** `02` is the no-language-model, raw-amino-acid floor — strong on the
random split, collapsing on the contiguous (region-holdout) split because it can only memorize positions
it trained on. `04` is the hand-engineered physicochemical control. `06` is the no-training control: a
frozen surprisal score read off the language model. This notebook asks the arm-3 question: **do learned
embedding features beat all three bounding baselines, especially on the contiguous split** — the evidence
that language-model representations carry signal beyond memorized positions and beyond the raw surprisal.

**Design (project decisions):**
- **Features** = ESM embedding deltas from `07a` (D035/D036): `delta_site`, `delta_pooled`, `delta_local`.
- **Reduction, D037 (non-negotiable).** The raw matrix is ~10k+ columns — feeding it whole into a tree
  invites leakage/overfitting under the hard splits. PCA is fit **inside each CV fold on the training
  positions only**, transformed onto the held-out fold. Never on the full data.
- **Classifiers, D023–D025**: XGBoost, random forest, L2 logistic regression. **No pre-committed winner
  (D026)** — selected on measured accuracy + calibration under the contiguous split.
- **Threshold**: Youden's J fit on the train fold only, applied to the held-out fold. No test label
  touches a prediction.
- **Metrics + stats** mirror `02`/`06` exactly: ROC-AUC, PR-AUC, balanced acc, F1, MCC, domain-weighted
  utility (TP+1, TN+1, FN−0.25, FP−1), bootstrap 95% CIs, DeLong, McNemar, dummy floor.

**Inputs read from disk** (never in-memory state, per CLAUDE.md):
- labels: `data/processed/traditional_ml_aa_identity/modeling_dataset.parquet`
- embeddings: `data/features/plm_embeddings/pllm_embeddings.parquet` — from the Colab twin
  `07a_pllm_embedding_extraction_colab.ipynb`; **run that first**. Absent → PREVIEW banner, clean stop.


In [ ]:
# self-contained: resolve project root via .projectroot, read from disk
import sys, os, json, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'.projectroot').exists())
sys.path.insert(0, str(root)); from paths import *

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
    balanced_accuracy_score, f1_score, matthews_corrcoef, confusion_matrix, roc_curve,
    precision_recall_curve)
from xgboost import XGBClassifier
import joblib

SEED = 42; N_SPLITS = 5
# >>> PROVISIONAL TRIMMED RUN (revisit) <<<  set for speed on a 16 GB laptop (~25 min vs ~45).
# TODO(revisit): restore PCA_K=50 and N_BOOT=2000 for the definitive run. The trim is NOT cheap:
# (1) K=25 leaves a lot of each delta_site block's variance uncaptured. Per the 07 pca_variance
#     table, EVERY delta_site block (the primary D035 feature) needs >100 PCs to reach 90% variance;
#     its TOP-10-PC variance ranges ~19-48% (ESM-C lowest at ~19-20%, ESM-1b/1v/2-150m ~44-48%), and
#     retained variance at K=25 is at least those figures. So 25 PCs still miss the majority of each
#     block's variance. Low-variance-but-predictive directions are especially at risk — the best
#     class-separating direction (ESM-C 600M PC1, single-feature AUC 0.82) carried only ~3% variance.
#     The trimmed run may understate the full-embedding model, esp. on the hard contiguous split.
# (2) 1000 bootstraps widen the CI noise floor, so close contiguous-split gaps may not reach
#     significance the 2000-resample run would resolve.
N_BOOT = 2000
PCA_K = 50            # components PER MODEL, fit inside each fold (D037). NOTE: delta_site blocks
                      # need >100 PCs for 90% variance (07 EDA), so 25 is a speed compromise that
                      # under-covers; 50-200 is the intended range for the definitive run.
BLOCKS = ["delta_site","delta_pooled","delta_local"]   # D035/D036 feature blocks to include
MODEL_ORDER = ["esm1b","esm1v","esm2_150m","esm2_650m","esm2_3b","esmc_300m","esmc_600m"]
MODEL_LABEL = {"esm1b":"ESM-1b 650M","esm1v":"ESM-1v 650M","esm2_150m":"ESM-2 150M",
               "esm2_650m":"ESM-2 650M","esm2_3b":"ESM-2 3B","esmc_300m":"ESM-C 300M","esmc_600m":"ESM-C 600M"}
MODEL_COLORS = {"logreg":"#2c6fbb","rf":"#2e8b57","xgboost":"#7a4fa3","dummy":"#9aa0a6"}
PAL = {"blue":"#2c6fbb","green":"#2e8b57","purple":"#7a4fa3","pink":"#b03060","grey":"#9aa0a6"}
sns.set_theme(style="whitegrid")

NBNAME = "08_pllm_supervised_benchmark"
FIGDIR = FIGURES/NBNAME; FIGDIR.mkdir(parents=True, exist_ok=True)
MODELDIR = MODELS/"pllm_supervised"; MODELDIR.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)
print("root:", root, "| seed:", SEED, "| PCA_K per model:", PCA_K)


## 1. Load labels + embeddings, and validate the join

If the embeddings parquet is missing, flip to **PREVIEW mode** (print what's needed, stop cleanly) so
the notebook is reviewable before the Colab extraction lands.

In [ ]:
df = pd.read_parquet(PROCESSED/"traditional_ml_aa_identity"/"modeling_dataset.parquet")
EMB_PATH = FEATURES_PLM_EMB/"pllm_embeddings.parquet"

PREVIEW = not EMB_PATH.exists()
if PREVIEW:
    print("="*78)
    print("PREVIEW MODE — embeddings not found at:", EMB_PATH)
    print("Run 1.5 - Colab notebooks/07a_pllm_embedding_extraction_colab.ipynb, download")
    print("out/pllm_embeddings.parquet, drop it at the path above, then re-run this notebook.")
    print("="*78)
else:
    emb = pd.read_parquet(EMB_PATH)
    # --- boundary checks (CLAUDE.md) ---
    assert df.seq_id.is_unique and emb.seq_id.is_unique, "duplicate seq_id"
    assert set(emb.seq_id) == set(df.seq_id), "embedding/label seq_id sets differ"
    assert len(emb) == len(df), f"row count mismatch {len(emb)} vs {len(df)}"
    data = df.merge(emb, on="seq_id", how="inner", validate="1:1").sort_values(
        ["position","mut_aa"]).reset_index(drop=True)
    y = data.DMS_score_bin.values.astype(int)
    pos = data.position.values
    feat_cols = [c for c in emb.columns if c != "seq_id"]
    present_models = [m for m in MODEL_ORDER if any(c.startswith(f"{m}__") for c in feat_cols)]
    def model_cols(m): return [c for c in feat_cols
                               if c.startswith(f"{m}__") and any(f"__{b}__" in c for b in BLOCKS)]
    MODEL_GROUPS = {m: model_cols(m) for m in present_models if model_cols(m)}
    assert MODEL_GROUPS, "no {model}__{block}__ columns for the requested BLOCKS"
    Xdf = data[[c for cols in MODEL_GROUPS.values() for c in cols]]
    X_all = Xdf.values.astype(np.float32)
    assert np.isfinite(X_all).all(), "non-finite embeddings"

    # leakage guard (CLAUDE.md / D037): no identity/label column; no dim tracks y almost perfectly.
    # chunked float32 per-feature |corr(x,y)| — O(N*D), never materializes a D×D matrix.
    used = [c for cols in MODEL_GROUPS.values() for c in cols]
    forbidden = ("seq_id","position","DMS","score","bin","wt_aa","mut_aa")
    assert not any(any(f in c for f in forbidden) for c in used), "identity/label column in features"
    yz = (y - y.mean()); yz = yz/ (np.linalg.norm(yz) + 1e-12)
    mx = 0.0
    for s in range(0, X_all.shape[1], 2048):
        blk = X_all[:, s:s+2048].astype(np.float32)
        blk = blk - blk.mean(0)
        nrm = np.linalg.norm(blk, axis=0) + 1e-12
        r = np.abs((blk * yz[:,None]).sum(0) / nrm)
        mx = max(mx, float(np.nanmax(r)))
    assert mx < 0.99, f"an embedding dim tracks the label almost perfectly (r={mx:.3f}) — leakage?"
    n_dims = X_all.shape[1]
    print(f"variants={len(data)} | class balance={np.bincount(y).tolist()} | "
          f"models={list(MODEL_GROUPS)} | raw feature dims={n_dims} | max|corr(x,y)|={mx:.3f}")


## 2. Three split schemes (identical to `02`/`06`)

The same `random` / `modulo` / `contiguous` 5-fold splits, seed 42 — reused verbatim so all four arms
compare on exactly the same partitions. `modulo` and `contiguous` hold out whole positions, so no
position leaks across train/test; `contiguous` withholds whole regions (hardest, most realistic).

In [ ]:
if not PREVIEW:
    def make_folds(data, n_splits=N_SPLITS, seed=SEED):
        idx = np.arange(len(data)); pos = data.position.values; folds={}
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        folds["random"] = list(skf.split(idx, data.DMS_score_bin.values))
        uniq = np.sort(data.position.unique())
        folds["modulo"] = [(idx[~np.isin(pos, uniq[uniq%n_splits==k])],
                            idx[ np.isin(pos, uniq[uniq%n_splits==k])]) for k in range(n_splits)]
        folds["contiguous"] = [(idx[~np.isin(pos,bk)], idx[np.isin(pos,bk)])
                               for bk in np.array_split(uniq, n_splits)]
        return folds
    folds = make_folds(data)
    for scheme in ["modulo","contiguous"]:
        for k,(tr,te) in enumerate(folds[scheme]):
            assert not (set(pos[tr]) & set(pos[te])), f"{scheme} fold {k} leaks a position"
    for scheme,fl in folds.items():
        tested=np.concatenate([te for _,te in fl]); assert len(np.unique(tested))==len(data)
        print(f"{scheme:11s} test sizes {[len(te) for _,te in fl]}")
    print("no position leakage in modulo/contiguous; full coverage in all schemes")


## 3. In-fold reduction (D037) — the guardrail

The raw embedding matrix is ~10k+ columns. Feeding it whole into a tree swamps the model and, under the
region-holdout splits, invites overfitting to the training positions. The reducer below fits
**StandardScaler + PCA per model on the training rows only**, then transforms the held-out rows — and it
is refit inside every fold. No test row ever influences the scaler mean or the PCA basis. Per-model PCA
(then concatenate) keeps each model's variance budget separate so a high-dim model can't dominate the
projection purely by width.

In [ ]:
if not PREVIEW:
    # D037 guardrail as plain functions so the fitted state (sklearn objects only) pickles cleanly
    # into the model bundle — a cell-defined class would not reload in a fresh scoring session.
    def fit_reducer(Xdf_tr, groups, k=PCA_K, seed=SEED):
        state={}
        for m, cols in groups.items():
            Z=Xdf_tr[cols].values.astype(np.float32)
            sc=StandardScaler().fit(Z)
            kk=min(k, Z.shape[1], Z.shape[0]-1)
            pc=PCA(n_components=kk, random_state=seed).fit(sc.transform(Z))
            state[m]={"cols":list(cols),"scaler":sc,"pca":pc}
        return state
    def apply_reducer(state, Xdf_any):
        return np.hstack([state[m]["pca"].transform(state[m]["scaler"].transform(
            Xdf_any[state[m]["cols"]].values.astype(np.float32))) for m in state])
    def reducer_dim(state): return int(sum(state[m]["pca"].n_components_ for m in state))

    # sanity: fit on a train fold, confirm width + that transform of held-out rows works
    _tr,_te = folds["contiguous"][0]
    _st = fit_reducer(Xdf.iloc[_tr], MODEL_GROUPS)
    _Ztr, _Zte = apply_reducer(_st, Xdf.iloc[_tr]), apply_reducer(_st, Xdf.iloc[_te])
    print(f"reducer OK: {len(MODEL_GROUPS)} models x up to {PCA_K} PCs -> {reducer_dim(_st)} reduced dims")
    print(f"  train fold {_Ztr.shape}, test fold {_Zte.shape}")


## 4. Models and evaluation

Three learners (D023–D025) plus a majority-class floor. The reduced PCA features are already
standardized, but logreg keeps a scaler in-pipeline for safety; trees are scale-invariant. Utility
weights match `02`/`04`/`06`: positive=functional, TP+1, TN+1, FN−0.25 (tolerable miss), FP−1 (costly
false resistance call).

In [ ]:
if not PREVIEW:
    UTIL = {"TP":1.0,"TN":1.0,"FN":-0.25,"FP":-1.0}
    def utility(y_true,y_pred):
        tn,fp,fn,tp = confusion_matrix(y_true,y_pred,labels=[0,1]).ravel()
        return (UTIL["TP"]*tp+UTIL["TN"]*tn+UTIL["FN"]*fn+UTIL["FP"]*fp)/len(y_true)
    def fit_threshold(proba_tr, y_tr):
        fpr,tpr,thr = roc_curve(y_tr, proba_tr); j=np.argmax(tpr-fpr); return thr[j]
    def metrics_at(y_true, proba, thr):
        pred=(proba>=thr).astype(int)
        return dict(roc_auc=roc_auc_score(y_true,proba), pr_auc=average_precision_score(y_true,proba),
            accuracy=accuracy_score(y_true,pred), balanced_acc=balanced_accuracy_score(y_true,pred),
            f1=f1_score(y_true,pred), mcc=matthews_corrcoef(y_true,pred), utility=utility(y_true,pred))
    def make_models():
        return {
            "logreg":  lambda: make_pipeline(StandardScaler(),
                            LogisticRegression(C=1.0, max_iter=5000, random_state=SEED)),
            "rf":      lambda: RandomForestClassifier(n_estimators=400, n_jobs=-1, random_state=SEED),
            "xgboost": lambda: XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.1,
                            subsample=0.9, colsample_bytree=0.9, eval_metric="logloss",
                            tree_method="hist", random_state=SEED, n_jobs=-1),
            "dummy":   lambda: DummyClassifier(strategy="most_frequent"),
        }
    print("models:", list(make_models().keys()), "| utility:", UTIL)


## 5. Train & evaluate — all models × all splits, reduction inside every fold

For each split and each fold: fit the reducer on the train rows only, transform both sides, fit the
learner on the reduced train features, predict the held-out fold. The decision threshold is Youden's J
on the **train-fold** predictions, applied to the test fold. Out-of-fold (OOF) probabilities are
collected so every variant is predicted exactly once per (model, split) for the ROC/PR curves and the
significance tests.

In [ ]:
if not PREVIEW:
    def run_all(folds, models):
        oof_proba, oof_pred, fold_metrics, thr_used = {}, {}, {}, {}
        for scheme, fl in folds.items():
            for mname, mk in models.items():
                key=(scheme,mname); oof=np.full(len(y),np.nan); pred=np.full(len(y),-1); per=[]; thrs=[]
                for tr,te in fl:
                    if mname=="dummy":
                        Ztr, Zte = np.zeros((len(tr),1)), np.zeros((len(te),1))
                    else:
                        st = fit_reducer(Xdf.iloc[tr], MODEL_GROUPS)
                        Ztr, Zte = apply_reducer(st, Xdf.iloc[tr]), apply_reducer(st, Xdf.iloc[te])
                    m=mk(); m.fit(Ztr, y[tr])
                    ptr = m.predict_proba(Ztr)[:,1] if hasattr(m,"predict_proba") else m.predict(Ztr).astype(float)
                    pte = m.predict_proba(Zte)[:,1] if hasattr(m,"predict_proba") else m.predict(Zte).astype(float)
                    thr = 0.5 if mname=="dummy" else fit_threshold(ptr, y[tr]); thrs.append(thr)
                    oof[te]=pte; pred[te]=(pte>=thr).astype(int); per.append(metrics_at(y[te],pte,thr))
                oof_proba[key]=oof; oof_pred[key]=pred
                fold_metrics[key]=pd.DataFrame(per); thr_used[key]=float(np.median(thrs))
        return oof_proba, oof_pred, fold_metrics, thr_used
    t0=time.time()
    oof_proba, oof_pred, fold_metrics, thr_used = run_all(folds, make_models())
    print(f"trained {len(oof_proba)} (model x split) combos with in-fold PCA in {time.time()-t0:.0f}s")


In [ ]:
if not PREVIEW:
    METRIC_KEYS=["roc_auc","pr_auc","accuracy","balanced_acc","f1","mcc","utility"]
    rows=[]
    for (scheme,mname),fm in fold_metrics.items():
        r={"split":scheme,"model":mname}
        for k in METRIC_KEYS:
            r[f"{k}_mean"]=round(fm[k].mean(),4); r[f"{k}_std"]=round(fm[k].std(),4)
        rows.append(r)
    grid=pd.DataFrame(rows).sort_values(["split","roc_auc_mean"],ascending=[True,False]).reset_index(drop=True)
    disp=["split","model","roc_auc_mean","pr_auc_mean","balanced_acc_mean","mcc_mean","utility_mean"]
    print("=== supervised-PLM metrics grid (mean over folds) ===")
    print(grid[disp].to_string(index=False))


## 6. Statistical significance (mirrors `02`/`06`)

Bootstrap 95% CIs on the pooled OOF predictions (2000×), DeLong tests comparing each model's ROC-AUC
against the best model per split, McNemar tests on the thresholded predictions, and a bootstrap test
that each model clears the majority-class dummy floor. Same estimators as the other arms so the
comparison is like-for-like.

In [ ]:
if not PREVIEW:
    def bootstrap_ci(y_true, proba, fn, n_boot=N_BOOT, seed=SEED):
        rng=np.random.default_rng(seed); n=len(y_true); vals=[]
        for _ in range(n_boot):
            idx=rng.integers(0,n,n)
            if len(np.unique(y_true[idx]))<2: continue
            vals.append(fn(y_true[idx], proba[idx]))
        return float(np.mean(vals)), float(np.percentile(vals,2.5)), float(np.percentile(vals,97.5))
    def _midrank(x):
        J=np.argsort(x); Z=x[J]; N=len(x); T=np.zeros(N); i=0
        while i<N:
            j=i
            while j<N and Z[j]==Z[i]: j+=1
            T[i:j]=0.5*(i+j-1)+1; i=j
        T2=np.empty(N); T2[J]=T; return T2
    def delong_test(y_true, p1, p2):
        yy=y_true.astype(int); posm=yy==1; negm=yy==0; m=posm.sum(); n=negm.sum()
        preds=np.vstack([p1,p2]); tz=np.array([_midrank(p) for p in preds])
        auc=np.array([(stats.rankdata(np.concatenate([p[posm],p[negm]]))[:m].sum()-m*(m+1)/2)/(m*n) for p in preds])
        v01=(tz[:,yy==1]-np.array([_midrank(p[posm]) for p in preds]))/n
        v10=1.0-(tz[:,yy==0]-np.array([_midrank(p[negm]) for p in preds]))/m
        sx=np.cov(v01); sy=np.cov(v10); s=sx/m+sy/n
        var=s[0,0]+s[1,1]-2*s[0,1]
        if var<=0: return auc[0],auc[1],1.0
        z=(auc[0]-auc[1])/np.sqrt(var); return float(auc[0]),float(auc[1]),float(2*stats.norm.sf(abs(z)))
    from statsmodels.stats.contingency_tables import mcnemar
    def mcnemar_test(y_true, pred1, pred2):
        a=pred1.astype(int); c=pred2.astype(int)
        n01=((a==y_true)&(c!=y_true)).sum(); n10=((a!=y_true)&(c==y_true)).sum()
        r=mcnemar([[0,int(n01)],[int(n10),0]], exact=False, correction=True)
        return int(n01), int(n10), float(r.pvalue)

    learners=[m for m in ["logreg","rf","xgboost"]]
    ci_rows=[]; sig_rows=[]
    for scheme in folds:
        aucs={m: float(grid[(grid.split==scheme)&(grid.model==m)].roc_auc_mean.values[0]) for m in learners}  # fold-mean, matches grid/four-way
        best=max(aucs, key=aucs.get)
        for m in learners+["dummy"]:
            p=oof_proba[(scheme,m)]
            ra,rlo,rhi=bootstrap_ci(y,p,roc_auc_score); pa,plo,phi=bootstrap_ci(y,p,average_precision_score)
            ci_rows.append(dict(split=scheme,model=m,roc_auc=ra,roc_lo=rlo,roc_hi=rhi,pr_auc=pa,pr_lo=plo,pr_hi=phi))
        for m in learners:
            if m==best:
                sig_rows.append(dict(split=scheme,model=m,vs_best=best,delong_p=np.nan,mcnemar_p=np.nan,
                                     vs_dummy_delong_p=np.nan,note="best")); continue
            _,_,dp=delong_test(y, oof_proba[(scheme,m)], oof_proba[(scheme,best)])
            _,_,mp=mcnemar_test(y, oof_pred[(scheme,m)], oof_pred[(scheme,best)])
            _,_,dpd=delong_test(y, oof_proba[(scheme,m)], oof_proba[(scheme,"dummy")])
            sig_rows.append(dict(split=scheme,model=m,vs_best=best,delong_p=dp,mcnemar_p=mp,vs_dummy_delong_p=dpd,note=""))
    ci=pd.DataFrame(ci_rows); sig=pd.DataFrame(sig_rows)
    print("best model per split:", {s:max({m:roc_auc_score(y,oof_proba[(s,m)]) for m in learners},key=lambda k:roc_auc_score(y,oof_proba[(s,k)])) for s in folds})
    print(ci[ci.model!="dummy"].round(3).to_string(index=False))


## 7. Model selection (D026) — decide on the hardest split, don't pre-commit

Per D026 the deployed classifier is chosen empirically, on the **contiguous** (region-holdout) split —
the hardest and most realistic — weighing ROC-AUC and calibration (Brier), not fixed in advance.

In [ ]:
if not PREVIEW:
    from sklearn.metrics import brier_score_loss
    sel_rows=[]
    for m in learners:
        p=oof_proba[("contiguous",m)]
        gm=grid[(grid.split=="contiguous")&(grid.model==m)]
        sel_rows.append(dict(model=m, roc_auc=float(gm.roc_auc_mean.values[0]),  # fold-mean, matches four-way
            utility=float(gm.utility_mean.values[0]),
            brier=brier_score_loss(y,np.clip(p,0,1))))
    sel=pd.DataFrame(sel_rows).sort_values(["roc_auc","utility"],ascending=False).reset_index(drop=True)
    SELECTED=sel.iloc[0].model
    print("selection on contiguous split (D026):"); print(sel.round(4).to_string(index=False))
    print(f"\n-> selected model: {SELECTED} (highest contiguous ROC-AUC; utility + calibration as tiebreak)")


## 8. The 4-way comparison (the payoff)

All four arms side by side, per split, best model within each arm:

| arm | notebook | what it uses | control type |
|---|---|---|---|
| AA-identity | `02` | raw amino-acid one-hot | no-language-model, supervised |
| physicochemical | `04` | hand-engineered descriptors | no-language-model, supervised (D027) |
| zero-shot pLLM | `06` | frozen ESM surprisal score | no-training (D027) |
| **supervised-PLM** | `08` (this) | **learned ESM embeddings** | **the arm-3 model** |

Each arm's metrics grid is read from `results/tables/` **if present**; a missing arm is simply omitted
(e.g. if an upstream notebook hasn't been run). The scientific reads to state next to the numbers:
- **vs zero-shot** — does *training on embeddings* beat the no-training score, especially on contiguous?
  If yes, learned features add signal beyond the raw surprisal.
- **vs AA-identity** — does the language model beat raw amino acids on the hard split where AA-identity
  collapsed? If supervised-PLM holds on contiguous, the representation is doing the work, not position
  memorization.
- **vs physicochemical** — does the LM beat hand-engineered chemistry?

If supervised-PLM does **not** clearly beat zero-shot on the hard split, that is a real, publishable
result too: it says the scalar surprisal already captures most of the usable signal for this protein and
the embedding complexity isn't justified.

In [ ]:
if not PREVIEW:
    # arm registry: label -> (grid csv stem, model-column name in that grid)
    ARMS = [
        ("aa_identity",   "02_traditional_ml_aa_identity_benchmark_metrics_grid.csv"),
        ("physicochem",   "04_physicochemical_benchmark_metrics_grid.csv"),
        ("zeroshot_pllm", "06_pllm_zeroshot_benchmark_metrics_grid.csv"),
    ]
    METR=[("roc_auc","ROC-AUC"),("pr_auc","PR-AUC"),("balanced_acc","bal.acc"),("mcc","MCC"),("utility","utility")]
    SPLITS=["random","modulo","contiguous"]

    def best_by_split(g, split, key):
        gg=g[(g.split==split)&(g.model!="dummy")]
        if not len(gg): return None
        row=gg.sort_values("roc_auc_mean",ascending=False).iloc[0]
        return float(row[f"{key}_mean"]), row.model

    # load other arms present on disk
    arm_grids={"supervised_pllm": grid}
    for label, stem in ARMS:
        p=TABLES/stem
        if p.exists(): arm_grids[label]=pd.read_csv(p)
        else: print(f"[note] {label} grid not on disk ({stem}) — omitted from comparison")

    cmp_rows=[]
    for split in SPLITS:
        for key,lab in METR:
            row={"split":split,"metric":lab}
            for arm,g in arm_grids.items():
                r=best_by_split(g, split, key)
                if r is not None: row[arm]=round(r[0],4); row[f"{arm}_model"]=r[1]
            cmp_rows.append(row)
    four_way=pd.DataFrame(cmp_rows)
    # headline: ROC-AUC across arms per split
    show_cols=["split","metric"]+[a for a in ["aa_identity","physicochem","zeroshot_pllm","supervised_pllm"] if a in four_way.columns]
    print("=== 4-way comparison (best model per arm per split) ===")
    print(four_way[four_way.metric=="ROC-AUC"][show_cols].to_string(index=False))

    # deltas vs the bounding baselines on the hard splits
    if "zeroshot_pllm" in four_way and "supervised_pllm" in four_way:
        hard=four_way[four_way.split.isin(["modulo","contiguous"])]
        wins=(hard["supervised_pllm"]>hard["zeroshot_pllm"]).sum()
        print(f"\nsupervised-PLM beats zero-shot on {wins}/{len(hard)} (metric x hard-split) cells")


## 9. Figures

`results/figures/08_pllm_supervised_benchmark/`, generic names, palette restricted to
blues/greens/purples/dark pinks.

In [ ]:
if not PREVIEW:
    learn=learners
    # 9a. ROC curves — one panel per split, one line per learner
    fig,axes=plt.subplots(1,3,figsize=(15,4.6),sharey=True)
    for ax,scheme in zip(axes,["random","modulo","contiguous"]):
        for m in learn:
            fpr,tpr,_=roc_curve(y,oof_proba[(scheme,m)]); a=roc_auc_score(y,oof_proba[(scheme,m)])
            ax.plot(fpr,tpr,color=MODEL_COLORS[m],lw=1.7,label=f"{m} ({a:.3f})")
        ax.plot([0,1],[0,1],ls="--",color=PAL["grey"],lw=1)
        ax.set_title(f"{scheme} split"); ax.set_xlabel("false positive rate")
        ax.legend(fontsize=8,loc="lower right",frameon=False)
    axes[0].set_ylabel("true positive rate")
    fig.suptitle("Supervised-PLM ROC by split",y=1.02)
    fig.tight_layout(); fig.savefig(FIGDIR/"roc_curves.pdf",bbox_inches="tight")
    fig.savefig(FIGDIR/"roc_curves.png",bbox_inches="tight",dpi=200); plt.show()

    # 9b. PR curves
    fig,axes=plt.subplots(1,3,figsize=(15,4.6),sharey=True); base=y.mean()
    for ax,scheme in zip(axes,["random","modulo","contiguous"]):
        for m in learn:
            pr,rc,_=precision_recall_curve(y,oof_proba[(scheme,m)]); a=average_precision_score(y,oof_proba[(scheme,m)])
            ax.plot(rc,pr,color=MODEL_COLORS[m],lw=1.7,label=f"{m} ({a:.3f})")
        ax.axhline(base,ls="--",color=PAL["grey"],lw=1)
        ax.set_title(f"{scheme} split"); ax.set_xlabel("recall")
        ax.legend(fontsize=8,loc="lower left",frameon=False)
    axes[0].set_ylabel("precision")
    fig.suptitle("Supervised-PLM precision-recall by split",y=1.02)
    fig.tight_layout(); fig.savefig(FIGDIR/"pr_curves.pdf",bbox_inches="tight")
    fig.savefig(FIGDIR/"pr_curves.png",bbox_inches="tight",dpi=200); plt.show()


In [ ]:
if not PREVIEW:
    schemes=["random","modulo","contiguous"]; x=np.arange(len(schemes))
    # 9c. ROC-AUC bars with bootstrap CIs
    fig,ax=plt.subplots(figsize=(10,4.8)); w=0.22
    for i,m in enumerate(learn):
        sub=ci[ci.model==m].set_index("split").reindex(schemes)
        err=np.vstack([sub.roc_auc-sub.roc_lo, sub.roc_hi-sub.roc_auc])
        ax.bar(x+i*w, sub.roc_auc, w, yerr=err, capsize=3, color=MODEL_COLORS[m],
               label=m, edgecolor="black", linewidth=.4)
    ax.axhline(0.5,ls="--",color=PAL["grey"],lw=1,label="no-skill")
    ax.set_xticks(x+w); ax.set_xticklabels(schemes); ax.set_ylim(0.4,1.0)
    ax.set_ylabel("ROC-AUC (95% bootstrap CI)"); ax.set_title("Supervised-PLM ROC-AUC by split and model")
    ax.legend(fontsize=8,ncol=4,frameon=False)
    fig.tight_layout(); fig.savefig(FIGDIR/"metric_bars.pdf",bbox_inches="tight")
    fig.savefig(FIGDIR/"metric_bars.png",bbox_inches="tight",dpi=200); plt.show()

    # 9d. utility bars
    fig,ax=plt.subplots(figsize=(10,4.6)); 
    for i,m in enumerate(learn):
        sub=grid[grid.model==m].set_index("split").reindex(schemes)
        ax.bar(x+i*w, sub.utility_mean, w, yerr=sub.utility_std, capsize=3,
               color=MODEL_COLORS[m], label=m, edgecolor="black", linewidth=.4)
    dv=[grid[(grid.split==s)&(grid.model=="dummy")].utility_mean.values[0] for s in schemes]
    ax.plot(x+w, dv, "k_", ms=16, label="dummy")
    ax.axhline(0,ls=":",color=PAL["grey"],lw=1)
    ax.set_xticks(x+w); ax.set_xticklabels(schemes)
    ax.set_ylabel("mean utility / prediction"); ax.set_title("Utility (TP+1, TN+1, FN-0.25, FP-1)")
    ax.legend(fontsize=8,ncol=4,frameon=False)
    fig.tight_layout(); fig.savefig(FIGDIR/"utility_bars.pdf",bbox_inches="tight")
    fig.savefig(FIGDIR/"utility_bars.png",bbox_inches="tight",dpi=200); plt.show()

    # 9e. confusion matrix per learner (panels across splits)
    for m in learn:
        fig,axes=plt.subplots(1,3,figsize=(12,3.8))
        for ax,scheme in zip(axes,schemes):
            cm=confusion_matrix(y,oof_pred[(scheme,m)],labels=[0,1])
            sns.heatmap(cm,annot=True,fmt="d",cmap="PuBuGn",cbar=False,ax=ax,
                        xticklabels=["non-func","func"],yticklabels=["non-func","func"])
            ax.set_title(scheme); ax.set_xlabel("predicted"); ax.set_ylabel("actual")
        fig.suptitle(f"Confusion matrices — {m}", y=1.03, fontsize=13)
        fig.tight_layout(); fig.savefig(FIGDIR/f"confusion_matrix_{m}.pdf",bbox_inches="tight")
        fig.savefig(FIGDIR/f"confusion_matrix_{m}.png",bbox_inches="tight",dpi=200); plt.show()


In [ ]:
if not PREVIEW:
    # 9f. the 4-way comparison figure — ROC-AUC across arms and splits (the headline)
    arm_cols=[("aa_identity",PAL["pink"],"AA-identity (02)"),
              ("physicochem",PAL["green"],"physicochem (04)"),
              ("zeroshot_pllm",PAL["blue"],"zero-shot pLLM (06)"),
              ("supervised_pllm",PAL["purple"],"supervised-PLM (08)")]
    present_arms=[(a,c,l) for a,c,l in arm_cols if a in four_way.columns]
    roc=four_way[four_way.metric=="ROC-AUC"].set_index("split").reindex(SPLITS)
    fig,ax=plt.subplots(figsize=(9,5)); x=np.arange(len(SPLITS)); w=0.8/max(len(present_arms),1)
    for i,(a,col,lab) in enumerate(present_arms):
        vals=roc[a].values.astype(float)
        ax.bar(x+i*w, vals, w, color=col, edgecolor="black", linewidth=.4, label=lab)
        for xi,v in zip(x+i*w, vals):
            if np.isfinite(v): ax.text(xi, min(v+.008,1.03), f"{v:.3f}", ha="center", fontsize=7)
    ax.axhline(0.5,ls="--",color=PAL["grey"],lw=1,label="no-skill")
    ax.set_xticks(x+w*(len(present_arms)-1)/2); ax.set_xticklabels(SPLITS)
    ax.set_ylabel("ROC-AUC (best model per arm)"); ax.set_ylim(0.45,1.08)
    ax.set_title("Four arms: AA-identity vs physicochem vs zero-shot vs supervised-PLM", pad=14)
    ax.legend(frameon=False, fontsize=8, ncol=2)
    fig.tight_layout(); fig.savefig(FIGDIR/"four_way_comparison.pdf",bbox_inches="tight")
    fig.savefig(FIGDIR/"four_way_comparison.png",bbox_inches="tight",dpi=200); plt.show()


## 10. Save tables and model bundles

Tables to `results/tables/`. Each learner is refit on the **full data** through a reducer fit on all
rows, and bundled per CLAUDE.md with everything needed to rebuild its input features: the fitted
per-model scalers + PCA bases, the column groups and order, the utility matrix, split metadata, the
selected-model flag, and its OOF metrics. Scoring a new variant means: extract the same embedding
blocks (07a) → apply this bundle's reducer → predict.

In [ ]:
if not PREVIEW:
    grid.to_csv(TABLES/f"{NBNAME}_metrics_grid.csv", index=False)
    ci.round(4).to_csv(TABLES/f"{NBNAME}_bootstrap_ci.csv", index=False)
    sig.round(5).to_csv(TABLES/f"{NBNAME}_significance.csv", index=False)
    sel.round(4).to_csv(TABLES/f"{NBNAME}_model_selection.csv", index=False)
    four_way.to_csv(TABLES/f"{NBNAME}_four_way_comparison.csv", index=False)
    pd.DataFrame([{"model":k[1],"split":k[0],"threshold":v} for k,v in thr_used.items()]
                 ).to_csv(TABLES/f"{NBNAME}_thresholds.csv", index=False)

    # refit reducer + each learner on ALL data, bundle for reuse (state = sklearn objects, pickles clean)
    red_state = fit_reducer(Xdf, MODEL_GROUPS)
    Zfull = apply_reducer(red_state, Xdf)
    for mname, mk in make_models().items():
        if mname=="dummy": continue
        m=mk(); m.fit(Zfull, y)
        bundle=dict(model=m, reducer_state=red_state, model_groups={k:list(v) for k,v in MODEL_GROUPS.items()},
                    blocks=BLOCKS, pca_k=PCA_K, reduced_dim=reducer_dim(red_state),
                    utility_matrix=UTIL, positive_class="functional(1)", n_splits=N_SPLITS, seed=SEED,
                    selected=(mname==SELECTED),
                    oof_metrics={s: fold_metrics[(s,mname)].mean().to_dict() for s in folds},
                    source="data/features/plm_embeddings/pllm_embeddings.parquet",
                    note="to score a new variant: extract same 07a blocks -> apply_reducer(reducer_state, Xdf) -> model.predict_proba")
        joblib.dump(bundle, MODELDIR/f"{mname}.joblib")
    print("saved tables:")
    for f in sorted(TABLES.glob(f"{NBNAME}_*.csv")): print(" ", f.name)
    print("saved model bundles:", [p.name for p in sorted(MODELDIR.glob("*.joblib"))])
    print("saved figures:")
    for f in sorted(FIGDIR.glob("*.pdf")): print(" ", f.name)
else:
    print("PREVIEW mode — nothing written. Drop the embeddings parquet in place and re-run.")


## 7b. Which embeddings carry the signal (per-model attribution)

The classifier trains on **all 7 models × all 3 delta blocks** (27,456 raw dims), reduced per model to
PCA components inside each fold (D037). That answers *what goes in* but not *which models the signal
comes from*. Two complementary views, both on the decisive **contiguous** region-holdout split:

- **Ablation** — train the selected learner on **one model's PCs alone** (7 single-model classifiers),
  giving a real held-out ROC-AUC per embedding model. "What can each model do by itself?"
- **Grouped importance** — sum the full model's feature importance (XGBoost gain; |coef| for logreg)
  over each model's PC block. "What does the ensemble actually lean on?" (captures complementarity a
  single-model ablation misses.)

In [ ]:
if not PREVIEW:
    ABLATE_SPLIT = "contiguous"; ABLATE_LEARNER = SELECTED  # SELECTED set by model-selection cell
    _q = four_way.query("split==@ABLATE_SPLIT and metric=='ROC-AUC'")
    FULL_AUC = float(_q["supervised_pllm"].iloc[0])
    fl = folds[ABLATE_SPLIT]
    abl_rows=[]
    for m, cols in MODEL_GROUPS.items():
        oof=np.full(len(y), np.nan)
        for tr,te in fl:
            st = fit_reducer(Xdf.iloc[tr], {m: cols})      # reduce THIS model's block only
            Ztr, Zte = apply_reducer(st, Xdf.iloc[tr]), apply_reducer(st, Xdf.iloc[te])
            clf = make_models()[ABLATE_LEARNER](); clf.fit(Ztr, y[tr])
            oof[te] = clf.predict_proba(Zte)[:,1]
        abl_rows.append(dict(model=m, model_label=MODEL_LABEL.get(m,m),
                             solo_roc_auc=round(roc_auc_score(y, oof),4),
                             n_pcs=int(min(PCA_K, len(cols))), raw_dims=len(cols)))
    ablation = pd.DataFrame(abl_rows).sort_values("solo_roc_auc", ascending=False).reset_index(drop=True)
    ablation.to_csv(TABLES/f"{NBNAME}_per_model_ablation.csv", index=False)
    print(f"per-model solo ROC-AUC on {ABLATE_SPLIT} ({ABLATE_LEARNER}); full 7-model model = {FULL_AUC:.4f}")
    print(ablation[["model_label","solo_roc_auc","n_pcs","raw_dims"]].to_string(index=False))


In [ ]:
if not PREVIEW:
    # grouped importance from the full model refit on ALL data through a full-data reducer
    st_full = fit_reducer(Xdf, MODEL_GROUPS)
    Zfull = apply_reducer(st_full, Xdf)
    pcs_per = [st_full[m]["pca"].n_components_ for m in st_full]           # PCs per model, in group order
    bounds = np.cumsum([0]+pcs_per)
    grp_models = list(st_full.keys())
    imp_model = {}
    clf = make_models()[ABLATE_LEARNER](); clf.fit(Zfull, y)
    if ABLATE_LEARNER == "xgboost":
        imp = clf.feature_importances_
    else:
        est = clf[-1] if hasattr(clf,"__getitem__") else clf
        if hasattr(est,"coef_"): imp = np.abs(est.coef_).ravel()
        else: imp = getattr(est,"feature_importances_", np.ones(Zfull.shape[1]))
    for gi,m in enumerate(grp_models):
        imp_model[m] = float(imp[bounds[gi]:bounds[gi+1]].sum())
    tot = sum(imp_model.values()) or 1.0
    importance = pd.DataFrame([dict(model=m, model_label=MODEL_LABEL.get(m,m),
                    importance_share=round(imp_model[m]/tot,4)) for m in grp_models]
                 ).sort_values("importance_share", ascending=False).reset_index(drop=True)
    importance.to_csv(TABLES/f"{NBNAME}_per_model_importance.csv", index=False)
    print(f"grouped {ABLATE_LEARNER} importance share by embedding model (full-data fit):")
    print(importance.to_string(index=False))


In [ ]:
if not PREVIEW:
    EMB_COLORS = {"esm1b":"#2c6fbb","esm1v":"#1f4e8c","esm2_150m":"#57a773",
                  "esm2_650m":"#2e8b57","esm2_3b":"#1d6b3f","esmc_300m":"#7a4fa3","esmc_600m":"#b03060"}
    order = ablation.model.tolist()
    fig, ax = plt.subplots(1, 2, figsize=(13,4.8))
    ax[0].bar([MODEL_LABEL[m] for m in order], ablation.set_index("model").loc[order,"solo_roc_auc"],
              color=[EMB_COLORS.get(m,'#2c6fbb') for m in order])
    ax[0].axhline(FULL_AUC, ls="--", color=PAL["pink"], label=f"all-7 model ({FULL_AUC:.3f})")
    ax[0].axhline(0.5, ls=":", color=PAL["grey"]); ax[0].set_ylim(0.45,1.0)
    ax[0].set_ylabel("solo ROC-AUC"); ax[0].set_title(f"Each model alone ({ABLATE_SPLIT}, {ABLATE_LEARNER})")
    ax[0].legend(frameon=False); ax[0].tick_params(axis="x", rotation=30)
    io = importance.model.tolist()
    ax[1].bar([MODEL_LABEL[m] for m in io], importance.set_index("model").loc[io,"importance_share"],
              color=[EMB_COLORS.get(m,'#2c6fbb') for m in io])
    ax[1].set_ylabel(f"{ABLATE_LEARNER} importance share"); ax[1].set_title("What the full model leans on")
    ax[1].tick_params(axis="x", rotation=30)
    for a in ax: a.set_xticklabels(a.get_xticklabels(), ha="right")
    fig.tight_layout(); fig.savefig(FIGDIR/"per_model_attribution.pdf",bbox_inches="tight")
    fig.savefig(FIGDIR/"per_model_attribution.png",bbox_inches="tight",dpi=200); plt.show()


## 11. Summary

Read the numbers off the tables above; the interpretation to state next to them:

- **Does supervised-PLM clear the dummy floor and hold across splits?** `metrics_grid` + `bootstrap_ci`
  + `significance` — ROC-AUC vs the 0.5 line, with DeLong/McNemar vs the best model per split.
- **The 4-way payoff.** `four_way_comparison` (table + `four_way_comparison.pdf`): supervised-PLM against
  AA-identity (02), physicochemical (04), and zero-shot pLLM (06), per split. The decisive cell is the
  **contiguous** split — the region-holdout where AA-identity collapses. If supervised-PLM stays high
  there and beats zero-shot, learned embedding features carry signal beyond both memorized positions and
  the raw surprisal. If it merely matches zero-shot, the scalar score already captures the usable signal
  and the embedding complexity isn't justified — a real result either way.
- **Which model (D026).** `model_selection` — chosen on contiguous ROC-AUC + calibration, not fixed in
  advance.

**Limitations, stated where the numbers live.**
- **D037 guardrail** — the raw ~10k-dim matrix is reduced by PCA fit *inside each fold* (train only); no
  test row informs the scaler or PCA basis, and no identity/label column enters the matrix (asserted in
  §1). The reported numbers are therefore free of the reduction-leakage that feeding raw embeddings would
  invite.
- **D039 (sequence-only blind spot)** — these embeddings, like the zero-shot scores, are shaped by
  foldability/stability and can under-detect a *catalytic-but-stable* knockout (e.g. an active-site
  substitution that abolishes activity without destabilizing the fold). Training on embeddings tests
  whether the supervised head recovers signal the zero-shot score misses; it does **not** close the
  stability-vs-catalysis gap. That is the job of the ESMFold structural-epistasis features (D015/D038,
  for the double mutants) and the wet-lab panel.

## 12. What supervising the pLLM buys us, and what it leads to next

The whole point of this arm is to read one sentence off Table 2: **when we supervise the pLLM, we
get ___** — and which word fills that blank decides the next experiment.

- **If supervised-PLM beats zero-shot on the contiguous split** → *when we supervise the pLLM, we
  get signal the frozen surprisal score missed*: the embedding carries functional information the
  scalar log-likelihood-ratio throws away, and training recovers it. **This leads us to feature
  fusion next** — combine the surprisal scalars (06, on disk) and the site-level score features
  (D032/D033: full 20-way logit vector, rank, margin, per-position entropy) *with* these embeddings
  in one classifier, same three-split battery and in-fold reducer, to measure how much the two
  families stack (they share the same 4,783 keys, so no new extraction is needed).
- **If supervised-PLM only matches zero-shot on the contiguous split** → *when we supervise the
  pLLM, we get nothing the scalar surprisal did not already give us*: the embedding complexity is
  not justified for this protein, and fusion would be redundant. **This leads us instead to change
  feature family** — the ESMFold structural-epistasis features (D015/D038), which is where a new
  signal would have to come from, not from more sequence-embedding machinery.

Either way the arm points past single-mutant sequence features:

- **Feature fusion** is the immediate, data-ready follow-up (reuses this notebook's scaffolding).
- **Structural / ESMFold (D015/D038)** is a separate regime, not a drop-in: those features (ΔpLDDT,
  predicted-structure perturbation of a double mutant vs WT and vs its constituent singles) are for
  **double / combinatorial mutants** (Deng 2012), where summed per-site sequence scores cannot
  represent epistasis. It needs the double-mutant label set and its own ESMFold extraction, so it is
  its own arm on its own data.
- Both feed the **wet-lab pAmpGent panel** — the loop-closer that tests whether *any* of these
  predictions track real function rather than DMS-structured labels, and where the D039
  catalytic-but-stable blind spot is finally checked against biology instead of held-out folds.